# Task 1: QLoRA fine-tuning of Qwen2.5-0.5B-Instruct (Colab T4)

**Before running:** upload these from the repo's `task1_lora/` folder to Google Drive at `MyDrive/tndrr/task1_lora/`:
`config.json`, `requirements-colab.txt`, `data/train.jsonl`, `data/test.jsonl`.

Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> Run all.

Outputs written to Drive (`MyDrive/tndrr/task1_lora/`):
- `adapter/` - the LoRA adapter + tokenizer
- `results/train_log.json` - every logged step (loss, lr, grad norm)
- `results/train_metrics.json` - runtime, final loss, peak GPU memory, library versions
- `results/train_loss.png` - loss curve

In [ ]:
# 1. Mount Drive and check the input files are there (fail early, before installing anything).
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/tndrr/task1_lora'
for f in ['config.json', 'requirements-colab.txt', 'data/train.jsonl']:
    assert os.path.exists(f'{BASE}/{f}'), f'missing {BASE}/{f} - upload it to Drive first'
os.makedirs(f'{BASE}/results', exist_ok=True)
print('inputs found in', BASE)

In [ ]:
# 2. Install pinned libraries. torch is left as Colab ships it (matched to Colab's CUDA).
!pip install -q -r /content/drive/MyDrive/tndrr/task1_lora/requirements-colab.txt

In [ ]:
# 3. Imports, config, seed, GPU check.
import json, time, torch, transformers, peft, trl, bitsandbytes
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

cfg = json.load(open(f'{BASE}/config.json'))
set_seed(cfg['seed'])

assert torch.cuda.is_available(), 'No GPU: set Runtime -> Change runtime type -> T4 GPU'
GPU = torch.cuda.get_device_name(0)
VERSIONS = {'torch': torch.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__,
            'trl': trl.__version__, 'bitsandbytes': bitsandbytes.__version__}
print(GPU, VERSIONS)

In [ ]:
# 4. Data -> prompt/completion text pairs.
# prompt = Qwen chat template up to and including '<|im_start|>assistant\n' (same as evaluate.ipynb uses).
# completion = the JSON label + '<|im_end|>' (end-of-turn, so the model learns to stop).
# TRL puts the loss ONLY on completion tokens (checked in cell 7). The prompt is pre-templated as a string
# on purpose: with chat-message format TRL 0.21 would also put loss on the '<|im_start|>assistant' header.
tokenizer = AutoTokenizer.from_pretrained(cfg['base_model'])
tokenizer.padding_side = 'right'   # right padding for training

def messages(row):
    return [{'role': 'system', 'content': cfg['system_prompt']},
            {'role': 'user', 'content': row['input']}]

def to_pair(row):
    return {
        'prompt': tokenizer.apply_chat_template(messages(row), tokenize=False, add_generation_prompt=True),
        'completion': json.dumps(row['output'], ensure_ascii=False) + tokenizer.eos_token,
    }

rows = [json.loads(l) for l in open(f"{BASE}/{cfg['data']['train']}", encoding='utf-8')]
train_ds = Dataset.from_list([to_pair(r) for r in rows])
print(len(train_ds), 'training examples')
print(train_ds[0])

In [ ]:
# 5. Base model in 4-bit NF4 + LoRA adapters.
# fp16 compute because the T4 has no bf16 support.
q = cfg['quantization']
bnb_config = BitsAndBytesConfig(
    load_in_4bit=q['load_in_4bit'],
    bnb_4bit_quant_type=q['bnb_4bit_quant_type'],
    bnb_4bit_use_double_quant=q['bnb_4bit_use_double_quant'],
    bnb_4bit_compute_dtype=getattr(torch, q['bnb_4bit_compute_dtype']),
)
model =AutoModelForCausalLM.from_pretrained(
    cfg['base_model'], quantization_config=bnb_config, torch_dtype=torch.float16, device_map={'': 0})
model.config.use_cache = False     # cache is only useful for generation

# LoRA is NOT applied here. SFTTrainer applies it exactly once (peft_config in cell 6), in this order:
#   prepare_model_for_kbit_training(use_gradient_checkpointing=True)  -> freezes base, casts norms to fp32,
#                                                                         makes inputs require grad
#   get_peft_model(lora_config)                                        -> adds trainable LoRA weights
# Why not do it ourselves: TRL 0.21 re-runs prepare_model_for_kbit_training on ANY 4-bit model it is given,
# and that call freezes every parameter - including LoRA weights added beforehand (-> 0 trainable params).
lora_config = LoraConfig(**cfg['lora'])

In [ ]:
# 6. Trainer.
t = cfg['training']
args = SFTConfig(
    output_dir='/content/out',            # local scratch; only the final adapter goes to Drive
    num_train_epochs=t['num_train_epochs'],
    learning_rate=t['learning_rate'],
    lr_scheduler_type=t['lr_scheduler_type'],
    warmup_ratio=t['warmup_ratio'],
    per_device_train_batch_size=t['per_device_train_batch_size'],
    gradient_accumulation_steps=t['gradient_accumulation_steps'],
    max_length=t['max_length'],
    fp16=t['fp16'],
    gradient_checkpointing=t['gradient_checkpointing'],          # TRL turns this on by default; set explicitly
    gradient_checkpointing_kwargs={'use_reentrant': False},      # non-reentrant version works with frozen base
    optim=t['optim'],
    logging_steps=t['logging_steps'],
    completion_only_loss=t['completion_only_loss'],
    packing=t['packing'],
    save_strategy='no',
    report_to='none',
    seed=cfg['seed'],
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, processing_class=tokenizer,
                     peft_config=lora_config)

In [ ]:
# 7. Check that the loss really is on answer tokens only.
# Labels of -100 are ignored by the loss; whatever is left must be just the JSON (+ end-of-turn token).
batch = next(iter(trainer.get_train_dataloader()))
n_unmasked = (batch['labels'] != -100).sum(dim=1).tolist()
print('label tokens NOT -100, per example in batch:', n_unmasked)
assert min(n_unmasked) > 0, 'an example has no answer tokens to learn from - masking is broken'

kept = batch['labels'][0][batch['labels'][0] != -100]
text = tokenizer.decode(kept)
print('example 0 tokens in input :', int((batch['input_ids'][0] != tokenizer.pad_token_id).sum()))
print('example 0 text with loss  :', repr(text))
assert text.startswith('{') and text.endswith('}' + tokenizer.eos_token), 'loss is not restricted to the answer - stop and fix'

In [ ]:
# 7b. Check the model the trainer will actually train: LoRA present, trainable, and in which dtype.
trainer.model.print_trainable_parameters()
name, p = next((n, p) for n, p in trainer.model.named_parameters() if 'lora_A' in n)
print('first lora_A param :', name)
print('dtype              :', p.dtype)
print('requires_grad      :', p.requires_grad)
n_trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
assert n_trainable > 0 and p.requires_grad, 'LoRA weights are frozen - do not train'
assert p.dtype == torch.float32, 'LoRA weights must be fp32 for the fp16 grad scaler'

In [ ]:
# 8. Train, then save the log and summary numbers to Drive.
torch.cuda.reset_peak_memory_stats()
start = time.time()
result = trainer.train()
wall = time.time() - start

log = trainer.state.log_history
json.dump(log, open(f'{BASE}/results/train_log.json', 'w'), indent=2)

summary = {
    'gpu': GPU,
    'versions': VERSIONS,
    'train_examples': len(train_ds),
    'total_steps': trainer.state.global_step,
    'train_runtime_sec': round(result.metrics['train_runtime'], 1),
    'wall_time_sec': round(wall, 1),
    'train_loss_mean': round(result.metrics['train_loss'], 4),
    'first_logged_loss': next(e['loss'] for e in log if 'loss' in e),
    'last_logged_loss': [e['loss'] for e in log if 'loss' in e][-1],
    'peak_gpu_mem_gb': round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    'trainable_params': sum(p.numel() for p in model.parameters() if p.requires_grad),
}
json.dump(summary, open(f'{BASE}/results/train_metrics.json', 'w'), indent=2)
print(json.dumps(summary, indent=2))

In [ ]:
# 9. Loss curve.
import matplotlib.pyplot as plt
pts = [(e['step'], e['loss']) for e in log if 'loss' in e]
plt.plot(*zip(*pts))
plt.xlabel('step'); plt.ylabel('training loss'); plt.title('QLoRA training loss')
plt.savefig(f'{BASE}/results/train_loss.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 10. Save the adapter (only the small LoRA weights, not the base model) to Drive.
trainer.model.save_pretrained(f'{BASE}/adapter')
tokenizer.save_pretrained(f'{BASE}/adapter')
!ls -la /content/drive/MyDrive/tndrr/task1_lora/adapter /content/drive/MyDrive/tndrr/task1_lora/results

In [ ]:
# 11. Smoke test on one TRAINING example (not a metric; the real test is evaluate.ipynb).
model.eval(); model.config.use_cache = True
ex = rows[0]
ids = tokenizer.apply_chat_template(messages(ex), add_generation_prompt=True, return_tensors='pt').to(0)
with torch.no_grad():
    out = model.generate(ids, max_new_tokens=cfg['eval']['max_new_tokens'], do_sample=False)
print('input   :', ex['input'])
print('expected:', json.dumps(ex['output'], ensure_ascii=False))
print('got     :', tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))